# `mdpp` Example: Secondary Structure (DSSP)

Per-residue secondary structure assignments and time evolution.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mplplots.utils import auto_ticks

from mdpp.analysis.dssp import compute_dssp
from mdpp.core.trajectory import load_trajectory

plt.style.use("mplplots.styles.GraphPadPrism")

In [ ]:
TOPOLOGY_PATH = Path("/path/to/topology.pdb")
TRAJECTORY_PATH = Path("/path/to/trajectory.xtc")
STRIDE = 5

if not TOPOLOGY_PATH.exists() or not TRAJECTORY_PATH.exists():
    raise FileNotFoundError(
        "Update TOPOLOGY_PATH and TRAJECTORY_PATH before running analysis cells."
    )

traj = load_trajectory(
    trajectory_path=TRAJECTORY_PATH,
    topology_path=TOPOLOGY_PATH,
    stride=STRIDE,
    atom_selection="protein",
)

print(f"Frames: {traj.n_frames}, Atoms: {traj.n_atoms}")

## Compute DSSP


In [ ]:
dssp = compute_dssp(traj, simplified=True)
print(f"Shape: {dssp.assignments.shape} (frames x residues)")
print(f"Categories: {dssp.categories}")

## Per-Residue Secondary Structure Frequency


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), dpi=120)
bottom = np.zeros(len(dssp.residue_ids))
colors = {"H": "tab:red", "E": "tab:blue", "C": "tab:gray"}
for i, cat in enumerate(dssp.categories):
    ax.bar(
        dssp.residue_ids,
        dssp.frequency[:, i],
        bottom=bottom,
        label=cat,
        color=colors.get(cat),
    )
    bottom += dssp.frequency[:, i]
ax.set_xlabel("Residue ID")
ax.set_ylabel("Fraction")
ax.set_title("Secondary Structure Frequency (DSSP)")
ax.legend()
auto_ticks(ax)
fig.tight_layout()

## Time Evolution Heatmap


In [ ]:
# Map categories to integers for heatmap
cat_to_int = {c: i for i, c in enumerate(dssp.categories)}
int_assignments = np.vectorize(cat_to_int.get)(dssp.assignments)

fig, ax = plt.subplots(figsize=(12, 4), dpi=120)
im = ax.imshow(
    int_assignments.T,
    aspect="auto",
    origin="lower",
    cmap="Set1",
    interpolation="none",
)
ax.set_xlabel("Frame")
ax.set_ylabel("Residue Index")
ax.set_title("DSSP Time Evolution")
cbar = fig.colorbar(im, ax=ax, ticks=range(len(dssp.categories)))
cbar.ax.set_yticklabels(dssp.categories)
fig.tight_layout()